# Caption & QA-Pair Generation

Two-stage generation pipeline used to build **BangladeshiVQA**, matching the paper's Dataset
Construction section. The two prompts below are reproduced verbatim from **Appendix A.3
(Prompt Templates)**.

1. **Stage 1 — Caption generation.** Gemini 3.5 Flash produces an objective Bangla caption
   and one of the seven dataset categories for each processed image.
2. **Stage 2 — QA-pair generation.** From each *human-validated* caption, Gemini 3.5 Flash
   drafts 2-4 Bangla question-answer pairs (text-only; no image is sent).

Both stages are followed by expert native-Bangla review (correction, filtering, and
re-labelling), so the raw model outputs here are **drafts**, not the released annotations.

Set `GOOGLE_API_KEY` in your environment (see `.env.example`) before running.

In [ ]:
import os
import json
import time

from google import genai
from google.genai import types
from json_repair import repair_json
from tqdm import tqdm

API_KEY = os.environ.get("GOOGLE_API_KEY")
if not API_KEY:
    raise ValueError("Set GOOGLE_API_KEY in your environment (see .env.example).")

client = genai.Client(api_key=API_KEY)
MODEL_NAME = "gemini-3.5-flash"

IMAGE_DIR = "images"            # processed 1024x1024 images, named CS_xxxx.jpg
CAPTION_PATH = "captions.json"  # Stage 1 output: image_id, category, caption
QA_PATH = "qa_pairs.json"       # Stage 2 output: image_id, q_type, question, answer

MAX_RETRIES = 5


def generate_json(contents, config=None):
    for _ in range(MAX_RETRIES):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME, contents=contents, config=config
            )
            if not response.text or not response.text.strip():
                raise ValueError("Empty response")
            return json.loads(repair_json(response.text))
        except Exception as error:
            time.sleep(20 if "429" in str(error) or "RESOURCE_EXHAUSTED" in str(error) else 3)
    return None

## Stage 1 - Caption generation

Image -> objective Bangla caption + one of the seven dataset categories. Results are saved
incrementally to `captions.json`; re-running resumes from where it stopped.

In [ ]:
CAPTION_PROMPT = """You are an advanced academic vision-annotation model optimized for Bengali datasets.
Analyze image {image_id} and strictly populate the JSON schema.

STRICT LANGUAGE AND SYSTEM BOUNDARIES:
- OUTPUT LANGUAGE: The 'caption' field MUST be written 100% in pure Bangla.
- NO MIXED OUTPUTS: Do NOT output English words, characters, or phrases in the caption text block.

CAPTION RULES:
1. PURELY OBJECTIVE: Describe only clear physical entities, clothing style, background items, and main actions.
2. NO INTUITIVE FLUFF: Absolutely zero emotional inferences. Completely exclude terms like 'গম্ভীর', 'শান্ত', 'উদ্বিগ্ন', 'প্রাণবন্ত', 'উৎসবমুখর', 'জরুরি অবস্থা', or 'মনে হয়'.
3. NO TIMESTAMPS: Ignore physical camera dates/times printed on the image layer entirely.
4. CONCISE LENGTH: Keep descriptions dense and clean."""

CAPTION_SCHEMA = {
    "type": "OBJECT",
    "properties": {
        "category": {
            "type": "STRING",
            "enum": [
                "Human Activities & Daily Life", "Culture & Arts", "Food & Cuisine",
                "Geography & Places", "Transportation", "Flora & Fauna", "Scene Text & Signs",
            ],
        },
        "caption": {"type": "STRING"},
    },
    "required": ["category", "caption"],
}


def generate_caption(image_id, image_bytes, mime_type):
    image_part = types.Part.from_bytes(data=image_bytes, mime_type=mime_type)
    prompt = CAPTION_PROMPT.replace("{image_id}", image_id)
    config = types.GenerateContentConfig(
        response_mime_type="application/json", response_schema=CAPTION_SCHEMA
    )
    result = generate_json([image_part, prompt], config=config)
    if not result:
        return None
    return {
        "image_id": image_id,
        "category": result.get("category", ""),
        "caption": result.get("caption", ""),
    }


captions = json.load(open(CAPTION_PATH, encoding="utf-8")) if os.path.exists(CAPTION_PATH) else []
done = {item["image_id"] for item in captions}
image_files = sorted(f for f in os.listdir(IMAGE_DIR) if f.lower().endswith((".jpg", ".jpeg", ".png")))

for filename in tqdm(image_files, desc="Captions"):
    image_id = os.path.splitext(filename)[0]
    if image_id in done:
        continue
    mime_type = "image/png" if filename.lower().endswith(".png") else "image/jpeg"
    with open(os.path.join(IMAGE_DIR, filename), "rb") as image_file:
        record = generate_caption(image_id, image_file.read(), mime_type)
    if record:
        captions.append(record)
        json.dump(captions, open(CAPTION_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=2)

print(f"Captions generated: {len(captions)}")

## Stage 2 - QA-pair generation

Runs on the **human-validated** captions from Stage 1 (text-only; no image is sent). Each
caption yields 2-4 Bangla QA pairs, saved incrementally to `qa_pairs.json`.

In [ ]:
QA_PROMPT = """Act as an expert native Bengali academic annotator for a VQA dataset.
You are given the verified ground-truth metadata of a scene:
- Image ID: {image_id}
- Ground-Truth Category: {category}
- Detailed Scene Caption: {caption}

Based STRICTLY and ONLY on the provided description, generate 2 to 4 highly precise, logical, and scene-specific Question-Answer (QA) pairs.

Return strictly a single raw JSON object with NO markdown code blocks or backticks. Follow this exact structure:
{
  "questions": [
    {
      "q_type": "Choose from [Existence, Attribute, OCR, Counting, Reasoning]",
      "question": "Natural, fluent Bengali question",
      "answer": "Strictly 1-3 words max Bengali answer"
    }
  ]
}

CRITICAL ACADEMIC BOUNDARIES:
1. NO HALLUCINATION: Do NOT invent entities, numbers, colors, or texts that are not explicitly mentioned in the caption. If the caption doesn't mention text, do NOT generate an OCR question.
2. Q_TYPE STRICTNESS:
   - OCR: Answer MUST be the exact literal text/numbers explicitly stated as written in the caption.
   - Counting: Answer MUST be a specific quantity (e.g., '৩টি', '৫জন'). No relative/vague words like 'কয়েকটি'.
   - Existence: Presence check (Answers: 'হ্যাঁ'/'না' or the short object name).
3. BENGALI SPELLING: Use Bangla Academy rules.
4. GRAMMAR: Use 'কি' strictly for Yes/No (হ্যাঁ/না) questions. Use 'কী' strictly for descriptive/informational questions. Ask directly like a human standing in the scene."""


def generate_qa(image_id, category, caption):
    prompt = (
        QA_PROMPT.replace("{image_id}", image_id)
        .replace("{category}", category)
        .replace("{caption}", caption)
    )
    result = generate_json(prompt)
    if not result:
        return None
    return [
        {
            "image_id": image_id,
            "q_type": entry.get("q_type"),
            "question": entry.get("question"),
            "answer": entry.get("answer"),
        }
        for entry in result.get("questions", [])
    ]


captions = json.load(open(CAPTION_PATH, encoding="utf-8"))  # human-validated captions
qa_pairs = json.load(open(QA_PATH, encoding="utf-8")) if os.path.exists(QA_PATH) else []
done = {pair["image_id"] for pair in qa_pairs}

for item in tqdm(captions, desc="QA pairs"):
    image_id = item["image_id"]
    if image_id in done:
        continue
    records = generate_qa(image_id, item["category"], item["caption"])
    if records:
        qa_pairs.extend(records)
        json.dump(qa_pairs, open(QA_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=2)

print(f"QA pairs generated: {len(qa_pairs)}")